# Figure 3: Combined national and regional outbreak summary

This notebook creates the three-panel figure used in the paper:

- **A:** National weekly epidemic curve.
- **B:** Total and affected communes by region.
- **C:** Outbreak-onset distribution by region.

Run all cells from the repository root or from this notebook. The input files are read from `6_metapopulation_model/data/`, and the generated PNG is written beside this notebook in `6_metapopulation_model/visualization/`.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

# Locate the repository even when the notebook is run from its own directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / "6_metapopulation_model" / "data").is_dir()),
    _here,
)
MODEL_DIR = REPO_ROOT / "6_metapopulation_model"
DATA_DIR = MODEL_DIR / "data"
FIGURES_DIR = MODEL_DIR / "visualization"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_theme(style="white")

In [ ]:
# Load the same repository inputs used by the metapopulation workflow.
outbreaks = pd.read_excel(
    DATA_DIR / "ppr_MA_2008_region_commune_agg.xlsx",
    sheet_name="Sheet1",
)
communes_status = pd.read_excel(DATA_DIR / "regions_communes_status.xlsx")

outbreaks["Start Date"] = pd.to_datetime(outbreaks["Start Date"])
outbreaks = outbreaks.sort_values("Start Date").reset_index(drop=True)

# National weekly totals.
outbreaks["week_start"] = outbreaks["Start Date"].dt.to_period("W-SUN").dt.start_time
weekly_totals = (
    outbreaks.groupby("week_start", as_index=False)
    .agg(
        cases=("Cases", "sum"),
        outbreaks=("Start Date", "size"),
    )
    .sort_values("week_start")
    .reset_index(drop=True)
)

# Total communes and affected communes by region.
status_unique = communes_status.drop_duplicates(subset=["REGION", "COMMUNE"])
total_by_region = status_unique.groupby("REGION").size().rename("total_communes")
affected_by_region = (
    outbreaks.drop_duplicates(subset=["REGION", "COMMUNE"])
    .groupby("REGION")
    .size()
    .rename("affected_communes")
)
region_summary = (
    total_by_region.to_frame()
    .join(affected_by_region, how="left")
    .fillna({"affected_communes": 0})
    .reset_index()
)

# Individual outbreak records and first/last outbreak dates by region.
timeline_events = outbreaks[["Start Date", "REGION"]].dropna().copy()
timeline_summary = timeline_events.groupby("REGION")["Start Date"].agg(
    first_on="min",
    last_on="max",
    outbreak_count="size",
)

# Regions with earlier outbreak onset appear first; unaffected regions follow.
region_order = timeline_summary.sort_values("first_on").index.tolist()
ALL_REGIONS = status_unique["REGION"].dropna().drop_duplicates().tolist()
ALL_REGIONS = sorted(ALL_REGIONS)

print(f"Loaded {len(outbreaks):,} outbreak records across {len(ALL_REGIONS)} regions.")
print(f"Writing figures to: {FIGURES_DIR}")

In [ ]:
# --- Combined 3-panel figure: National epidemic curve + Fig 4 + Fig 7 ---

# Fallbacks in case the notebook was run out of order
try:
    combined_region_order
except NameError:
    uninfected_regions = [r for r in ALL_REGIONS if r not in region_order]
    combined_region_order = region_order + uninfected_regions

try:
    region_summary_ordered
except NameError:
    region_summary_ordered = region_summary.set_index("REGION").reindex(combined_region_order).reset_index()
    region_summary_ordered["pct_affected"] = (
        region_summary_ordered["affected_communes"] / region_summary_ordered["total_communes"] * 100
    )

# Top panel data
x_dates = weekly_totals["week_start"]

# Bottom-right panel helper
timeline_events["start_num"] = mdates.date2num(timeline_events["Start Date"])

fig = plt.figure(figsize=(12.4, 9.2), dpi=300)
gs = fig.add_gridspec(
    2, 2,
    height_ratios=[1.05, 1.35],
    hspace=0.34,
    wspace=0.14
)

ax_top = fig.add_subplot(gs[0, :])
ax_left = fig.add_subplot(gs[1, 0])
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_left)

# ============================================================
# A — National Epidemic Curve
# ============================================================
ax_top.plot(x_dates, weekly_totals["cases"], color="#fa0202", linewidth=1.2, zorder=2, label="Cases")

poly = ax_top.fill_between(x_dates, weekly_totals["cases"], color="none")
x_min, x_max = mdates.date2num(x_dates.min()), mdates.date2num(x_dates.max())
y_max = weekly_totals["cases"].max()
grad = np.atleast_2d(np.linspace(0, 1, 256)).T
cmap_fill = mpl.colors.LinearSegmentedColormap.from_list("grad", ["#ffffff", "#fa0202"])
im = ax_top.imshow(
    grad, origin="lower", aspect="auto",
    extent=(x_min, x_max, 0, y_max),
    cmap=cmap_fill, alpha=0.6, zorder=1
)
im.set_clip_path(poly.get_paths()[0], transform=ax_top.transData)

ax_top.set_ylabel("Cases", color="#fa0202", fontweight="bold")
ax_top.tick_params(axis="y", colors="#fa0202")
ax_top.set_ylim(0, weekly_totals["cases"].max() * 1.2)

ax_top_twin = ax_top.twinx()
ax_top_twin.plot(
    x_dates, weekly_totals["outbreaks"],
    color="#0886da", linewidth=1.5, linestyle="--", alpha=0.8, label="Outbreaks"
)
ax_top_twin.set_ylabel("Outbreaks", color="#0886da", fontweight="bold")
ax_top_twin.tick_params(axis="y", colors="#0886da")
ax_top_twin.set_ylim(0, weekly_totals["outbreaks"].max() * 1.2)

peak_case_row = weekly_totals.loc[weekly_totals["cases"].idxmax()]
ax_top.annotate(
    peak_case_row["week_start"].strftime("%d %b"),
    xy=(peak_case_row["week_start"], peak_case_row["cases"]),
    xytext=(-15, 15), textcoords="offset points",
    ha="right", va="bottom", color="#fa0202", fontweight="bold", fontsize=9,
    arrowprops=dict(arrowstyle="->", color="#fa0202", linewidth=1.1, alpha=0.8)
)

peak_outbreak_row = weekly_totals.loc[weekly_totals["outbreaks"].idxmax()]
ax_top_twin.annotate(
    peak_outbreak_row["week_start"].strftime("%d %b"),
    xy=(peak_outbreak_row["week_start"], peak_outbreak_row["outbreaks"]),
    xytext=(15, 15), textcoords="offset points",
    ha="left", va="bottom", color="#0886da", fontweight="bold", fontsize=9,
    arrowprops=dict(arrowstyle="->", color="#0886da", linewidth=1.1, alpha=0.8)
)

h1, l1 = ax_top.get_legend_handles_labels()
h2, l2 = ax_top_twin.get_legend_handles_labels()
ax_top.legend(h1 + h2, l1 + l2, loc="upper right", frameon=False, fontsize=8)

ax_top.grid(False)
ax_top.grid(axis="x", linestyle=":", alpha=0.4)
ax_top.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax_top.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax_top.set_xlabel("Epidemiological Week (2008)")
ax_top.text(-0.02, 1.05, "A", transform=ax_top.transAxes, fontsize=11, fontweight="bold", va="bottom", ha="left")

# ============================================================
# B — Fig 4: Commune status by region
# ============================================================
COLOR_BAR_TOTAL = "#ffffff"
COLOR_BAR_AFFECTED = "#fa0202"

ax_left.barh(
    region_summary_ordered["REGION"],
    region_summary_ordered["total_communes"],
    color=COLOR_BAR_TOTAL, edgecolor="#888888", linewidth=0.7,
    label="Total communes"
)
affected_bars = ax_left.barh(
    region_summary_ordered["REGION"],
    region_summary_ordered["affected_communes"],
    color=COLOR_BAR_AFFECTED, edgecolor="#888888", linewidth=0.5,
    label="Affected communes", zorder=3
)

labels = [
    f"{int(a)}/{int(t)} ({pct:.0f}%)" if pd.notnull(a) and a > 0 else ""
    for a, t, pct in zip(
        region_summary_ordered["affected_communes"],
        region_summary_ordered["total_communes"],
        region_summary_ordered["pct_affected"],
    )
]
ax_left.bar_label(affected_bars, labels=labels, padding=3, fontsize=5, color="#000000")

for i, (total, affected) in enumerate(zip(
    region_summary_ordered["total_communes"],
    region_summary_ordered["affected_communes"],
)):
    if pd.notnull(total) and (pd.isnull(affected) or affected == 0):
        ax_left.text(total + 1.0, i, f"0/{int(total)} (0%)", va="center", fontsize=5, color="#000000")

ax_left.set_xlim(right=region_summary_ordered["total_communes"].max() * 1.15)
ax_left.set_xlabel("Number of communes")
ax_left.set_ylabel("Region")
ax_left.grid(False)
ax_left.grid(axis="x", linestyle=":", alpha=0.4)
ax_left.legend(loc="lower right", frameon=False, fontsize=6)
ax_left.invert_yaxis()
ax_left.text(-0.18, 1.05, "B", transform=ax_left.transAxes, fontsize=11, fontweight="bold", va="bottom", ha="left")

# ============================================================
# C — Fig 7: Onset distribution per region
# ============================================================
COLOR_VIOLIN_EDGE = "#fa0202"
CIRCLE_COLOR = "#4b94c7"

violin_colors = sns.light_palette(COLOR_VIOLIN_EDGE, n_colors=len(combined_region_order), reverse=True)

sns.violinplot(
    data=timeline_events,
    x="start_num", y="REGION",
    order=combined_region_order,
    inner=None, cut=0, bw_method=0.2,
    linewidth=0.6, palette=violin_colors, ax=ax_right, zorder=1
)

for collection in ax_right.collections:
    if isinstance(collection, mpl.collections.PolyCollection):
        collection.set_edgecolor(COLOR_VIOLIN_EDGE)
        collection.set_alpha(0.3)

sns.stripplot(
    data=timeline_events,
    x="start_num", y="REGION",
    order=combined_region_order,
    size=2.8, alpha=0.85,
    edgecolor="white", linewidth=0.3,
    color=CIRCLE_COLOR, ax=ax_right, zorder=3
)

for region, row in timeline_summary.iterrows():
    if region in combined_region_order:
        y = combined_region_order.index(region)
        ax_right.hlines(
            y, mdates.date2num(row["first_on"]), mdates.date2num(row["last_on"]),
            color=COLOR_VIOLIN_EDGE, linewidth=0.8, alpha=0.8, zorder=2
        )
        ax_right.text(
            mdates.date2num(row["last_on"]) + 4, y, f"(n={int(row['outbreak_count'])})",
            va="center", ha="left", fontsize=6, color="#000000", zorder=4
        )

ax_right.set_xlim(
    left=mdates.date2num(timeline_events["Start Date"].min()) - 10,
    right=mdates.date2num(timeline_events["Start Date"].max()) + 20
)
ax_right.xaxis_date()
ax_right.xaxis.set_major_locator(mdates.WeekdayLocator(interval=3))
ax_right.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax_right.set_xlabel("Outbreak start date (2008)")
ax_right.set_ylabel("")
ax_right.tick_params(axis="y", left=False, labelleft=False)
ax_right.grid(False)
ax_right.grid(axis="x", linestyle=":", alpha=0.25)
plt.setp(ax_right.get_xticklabels(), rotation=45, ha="right", fontsize=6)

legend_elements_b = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CIRCLE_COLOR, markersize=4,
           alpha=0.85, label="Individual outbreak record"),
    Line2D([0], [0], color=COLOR_VIOLIN_EDGE, linewidth=0.8, alpha=0.8,
           label="Region outbreak period (first-last)")
]
ax_right.legend(handles=legend_elements_b, loc="lower right", frameon=False, fontsize=6)
ax_right.text(-0.02, 1.05, "C", transform=ax_right.transAxes, fontsize=11, fontweight="bold", va="bottom", ha="left")

fig.tight_layout()

base = FIGURES_DIR / "Figure_3_combined_national_regional"
fig.savefig(
    f"{base}.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.08,
)

plt.show()
print(f"Saved: {base}.png")